### Data Cleaning & Processing
In this step, we are going to load the dataframe from csv file and do some cleaning and do some processing. 


#### Understanding the columns 
First section, we are going to load csv file into dataframe and discover columns, data types


In [15]:
import pandas as pd

#Load dataframe from csv file
file_path = "dataset/ratebeer_dataset_raw.csv"
df = pd.read_csv(file_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2924163 entries, 0 to 2924162
Data columns (total 13 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   beer_name           object
 1   beer_id             int64 
 2   brewer_id           int64 
 3   beer_abv            object
 4   beer_style          object
 5   review_appearance   object
 6   review_aroma        object
 7   review_palate       object
 8   review_taste        object
 9   review_overall      object
 10  review_time         int64 
 11  review_profilename  object
 12  review_text         object
dtypes: int64(3), object(10)
memory usage: 290.0+ MB


#### Check if empty values in the important columns
In the next section, you can find removing empty values for the important columns which will be used in model.
It looks like all the values are present

In [16]:
def check_empty_values_on_specific_columns():
    # Check for empty values in specific columns
    empty_beer_name = df[df['beer_name'].isnull() | (df['beer_name'] == '')]
    empty_beer_style = df[df['beer_style'].isnull() | (df['beer_style'] == '')]
    empty_review_profilename = df[df['review_profilename'].isnull() | (df['review_profilename'] == '')]
    empty_beer_id = df[df['beer_id'].isnull()]
    empty_brewer_id = df[df['brewer_id'].isnull()]
    empty_beer_abv = df[df['beer_abv'].isnull() | (df['beer_abv'] == '')]
    empty_review_time = df[df['review_time'].isnull()]
    empty_review_overall = df[df['review_overall'].isnull() | (df['review_overall'] == '')]
    empty_review_aroma = df[df['review_aroma'].isnull() | (df['review_aroma'] == '')]
    empty_review_palate = df[df['review_palate'].isnull() | (df['review_palate'] == '')]
    empty_review_taste = df[df['review_taste'].isnull() | (df['review_taste'] == '')]
    empty_review_appearance = df[df['review_appearance'].isnull() | (df['review_appearance'] == '')]
    # Print the count of empty values in each column
    print("Empty beer_name count:", len(empty_beer_name))
    print("Empty beer_style count:", len(empty_beer_style))
    print("Empty review_profilename count:", len(empty_review_profilename))
    print("Empty beer_id count:", len(empty_beer_id))
    print("Empty brewer_id count:", len(empty_brewer_id))
    print("Empty beer_abv count:", len(empty_beer_abv))
    print("Empty review_time count:", len(empty_review_time))
    print("Empty review_overall count:", len(empty_review_overall))
    print("Empty review_aroma count:", len(empty_review_aroma))
    print("Empty review_palate count:", len(empty_review_palate))
    print("Empty review_taste count:", len(empty_review_taste))
    print("Empty review_appearance count:", len(empty_review_appearance))
check_empty_values_on_specific_columns()

Empty beer_name count: 0
Empty beer_style count: 0
Empty review_profilename count: 0
Empty beer_id count: 0
Empty brewer_id count: 0
Empty beer_abv count: 0
Empty review_time count: 0
Empty review_overall count: 0
Empty review_aroma count: 0
Empty review_palate count: 0
Empty review_taste count: 0
Empty review_appearance count: 0


#### Using correct data types - beer_abv
beer_abv is alcohol rate of beer. It has to be float but now it is string. For errors, we are skipping and checking what to do in next section.

In [17]:
df['beer_abv'] = pd.to_numeric(df['beer_abv'], errors='coerce')
empty_beer_abv = df[df['beer_abv'].isnull() | (df['beer_abv'] == '')]
print("Empty beer_abv count:", len(empty_beer_abv))
distinct_empty_beer_id_count = empty_beer_abv['beer_id'].nunique()
print("Distinct count of empty 'beer_id':", distinct_empty_beer_id_count)
#There are 138K beer reviews without abv affecting around 23K beers
#Let's see if these beers are highly rated
reviews_per_beer = empty_beer_abv.groupby('beer_id').size().reset_index(name='number_of_reviews').sort_values(by='number_of_reviews', ascending=False)
# Display the result
reviews_per_beer.head(10)

Empty beer_abv count: 138637
Distinct count of empty 'beer_id': 23120


,beer_id,number_of_reviews
67,837,619
12931,88489,364
10946,74088,315
1586,11428,300
8413,55883,296
13010,88982,290
12930,88479,247
10556,71621,222
13628,93186,218
1215,8747,216


#### Handling null beer_abv values
There are quite big amount of beers having good amount reviews and missing abv. 

We can use preprocess by taking an average of the same style of beers that has a good accuracy in some examples

In [18]:
# there are some null values, fill with average per beer style abv
beer_style_abv_means = df.groupby('beer_style')['beer_abv'].transform('mean')
df['beer_abv'] = df['beer_abv'].fillna(beer_style_abv_means)
#finally fill with average values if still empty 
df['beer_abv'] = df['beer_abv'].fillna(df['beer_abv'].mean())
# do not continue in case it cannot fill the values
assert not df['beer_abv'].isnull().any(), f"Error: The column 'beer_abv' contains null values."

In [19]:
df[df['beer_id'] == 11428]
# Beer abv is calculated 4.68 and it is normally 4.6 which is not bad

,beer_name,beer_id,brewer_id,beer_abv,beer_style,review_appearance,review_aroma,review_palate,review_taste,review_overall,review_time,review_profilename,review_text
1382465,Rainier,11428,10946,4.68878,Pale Lager,3/5,3/10,3/5,3/10,7/20,1155686400,beerbill,"UPDATED: JAN 24, 2007 12 oz. can. I remember e..."
1382466,Rainier,11428,10946,4.68878,Pale Lager,2/5,3/10,3/5,3/10,6/20,1155427200,Delirium,"UPDATED: OCT 20, 2008 Bottle. Pours yellow wi..."
1382467,Rainier,11428,10946,4.68878,Pale Lager,2/5,2/10,2/5,3/10,11/20,1153267200,DuffMan,Nothing beats getting shitcanned around an ope...
1382468,Rainier,11428,10946,4.68878,Pale Lager,2/5,4/10,3/5,5/10,9/20,1152316800,linc33,Can provided by R. McCarty. Pours golden. Corn...
1382469,Rainier,11428,10946,4.68878,Pale Lager,3/5,3/10,3/5,6/10,9/20,1151798400,Saarlander,An old school lager that was drinkable. I actu...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1533650,Rainier,11428,75,4.68878,Pale Lager,1/5,1/10,2/5,3/10,8/20,1170633600,Cranberryboy,A good beer for drinking in large quantities. ...
1533651,Rainier,11428,75,4.68878,Pale Lager,2/5,1/10,2/5,2/10,3/20,1169683200,pintbypint,355ml can. Gold colour. Faint trace of white b...
1533652,Rainier,11428,75,4.68878,Pale Lager,2/5,2/10,1/5,2/10,3/20,1160956800,kamish,nope. maybe if i was born in 1967 or so. im gu...
1533653,Rainier,11428,75,4.68878,Pale Lager,2/5,3/10,2/5,3/10,5/20,1160092800,duffman462,"Well, I have a bad cold right now, so I figure..."


#### Ratings transformation
Ratings are stored in review_overall, review_arome, review_palate, review_taste and review_appearance columns.

These columns are string type with format 7/20, 3/5, 6/10, 4/5.

In the next section, we transformed it into float value for 5 base ratings. This is important for the ranking model 

In [20]:
#Create a generic method to handle all these values
def convert_review_column_to_float(df, column_name):
    # Extract the numerator and denominator from the specified column
    df[[f'{column_name}_numerator', f'{column_name}_denominator']] = df[column_name].str.split('/', expand=True)
    # Convert the extracted values to numeric
    df[f'{column_name}_numerator'] = pd.to_numeric(df[f'{column_name}_numerator'], errors='coerce')
    df[f'{column_name}_denominator'] = pd.to_numeric(df[f'{column_name}_denominator'], errors='coerce')
    # Calculate the converted float value for the specified column
    df[f'{column_name}'] = df[f'{column_name}_numerator'] / df[f'{column_name}_denominator'] * 5.0
    # Drop the intermediate columns if needed
    df.drop([f'{column_name}_numerator', f'{column_name}_denominator'], axis=1, inplace=True)
    return df

df = convert_review_column_to_float(df, 'review_overall')
df = convert_review_column_to_float(df, 'review_aroma')
df = convert_review_column_to_float(df, 'review_palate')
df = convert_review_column_to_float(df, 'review_taste')
df = convert_review_column_to_float(df, 'review_appearance')

check_empty_values_on_specific_columns()

Empty beer_name count: 0
Empty beer_style count: 0
Empty review_profilename count: 0
Empty beer_id count: 0
Empty brewer_id count: 0
Empty beer_abv count: 0
Empty review_time count: 0
Empty review_overall count: 0
Empty review_aroma count: 0
Empty review_palate count: 0
Empty review_taste count: 0
Empty review_appearance count: 0


In [21]:
#Change review_profilename to review_username - just for better readibility for us
df = df.rename(columns={'review_profilename': 'review_username'})

#### Reduce sparsity of matrix
For recommendation system, one of the biggest challange is sparsity. For our project, the matrix is composed by beers and users. 

As you can imagine, when dealing with 2,9 million reviews, coming up with a good suggestion is quite challanging. 

For better accuracy we have to remove beers that not have so many ratings and also usernames who did not review many beers

In [22]:
#Remove records when a beer does not have more than 50 reviews
df = df.groupby('beer_id').filter(lambda x: len(x) > 50)

#Remove records with a user when there are less than 50 reviews
username_review_counts = df['review_username'].value_counts()
valid_usernames = username_review_counts[username_review_counts > 50].index
df = df[df['review_username'].isin(valid_usernames)]

#### Creating beer dataframe
This is necessary as we need to have the list of beers. 

The beer dataframe will have the characteristics. It will be used also for embedding layer. 

We make sure, it does not have any data inconsistency. One example having multiple brewer_id for same beer_id. This is also handled in the next cell.

In [23]:
# Create a new dataframe to show only beers rated
df_beers = df[['beer_id', 'beer_name', 'beer_style', 'beer_abv', 'brewer_id']].drop_duplicates()
# Check any duplicates in beer_id 
duplicates_count = df_beers['beer_id'].duplicated().sum()
print("Count of duplicated 'beer_id' values:", duplicates_count)

# Not so many let's check
df_beers[df_beers['beer_id'].duplicated()]
# One example
print(df_beers[df_beers['beer_id'] == 44801])
# Brewer id is changed but they are same beer 

# Find the index of the latest 'brewer_id' for each 'beer_id'
latest_brewer_index = df_beers.groupby('beer_id')['brewer_id'].idxmax()

# Keep only the rows with the latest 'brewer_id' for each 'beer_id'
df_beers = df_beers.loc[latest_brewer_index]
# Check again duplicate beers
duplicates_count = df_beers['beer_id'].duplicated().sum()
# do not continue in case there are duplicates
assert duplicates_count == 0, f"Error: There are duplicate beer ids in dataset."

#Make sure this is also reflected into the original dataframe
df = pd.merge(df, df_beers[['beer_id', 'beer_name', 'beer_style', 'beer_abv', 'brewer_id']], on='beer_id', how='left', suffixes=('', '_new'))
# Update columns in 'df' with the new values
df['beer_name'] = df['beer_name_new'].combine_first(df['beer_name'])
df['beer_style'] = df['beer_style_new'].combine_first(df['beer_style'])
df['beer_abv'] = df['beer_abv_new'].combine_first(df['beer_abv'])
df['brewer_id'] = df['brewer_id_new'].combine_first(df['brewer_id'])
# Drop the temporary columns
df = df.drop(columns=['beer_name_new', 'beer_style_new', 'beer_abv_new', 'brewer_id_new'])

Count of duplicated 'beer_id' values: 501
        beer_id            beer_name beer_style  beer_abv  brewer_id
192284    44801  Bass &#40;Cask&#41;     Bitter       4.4         56
497295    44801  Bass &#40;Cask&#41;     Bitter       4.4      12418


#### Matrix Sparsity 
In the context of recommendation systems, matrix sparsity refers to the phenomenon where the user-item interaction matrix (like users rating items) contains a large proportion of missing or unobserved entries. This situation is common because in real-world scenarios, most users interact with or rate only a tiny fraction of the total available items. 

In our example, in a beer recommendation system with thousands of beers and users, any individual user will likely have seen and rated only a small subset of all beers. The sparsity percentage is 95! 

Consequently, the user-beer rating matrix will have many empty entries, indicating the beers that each user has not rated. 

Sparsity is a critical challenge in building effective recommendation systems. Sparse matrices make it difficult to accurately predict user preferences for the vast majority of items because there's little to no direct interaction data.

The print_sparsity_percentage function calculates and prints the sparsity percentage of a user-item interaction matrix in a recommendation system.

In [24]:
def print_sparsity_percentage(df, df_beers):    
    number_of_beers = df_beers['beer_id'].nunique()
    number_of_usernames = df['review_username'].nunique()
    total_possible_entries = number_of_beers * number_of_usernames
    number_of_ratings = df.shape[0]

    number_of_present_entries = number_of_ratings
    number_of_missing_entries = total_possible_entries - number_of_present_entries

    sparsity_percentage = (number_of_missing_entries / total_possible_entries) * 100
    print("Number of Beers:", number_of_beers)
    print("Number of Usernames:", number_of_usernames)
    print("Number of Ratings:", number_of_ratings)
    print("Sparsity Percentage:", sparsity_percentage)

print_sparsity_percentage(df, df_beers)

Number of Beers: 9276
Number of Usernames: 4560
Number of Ratings: 2006599
Sparsity Percentage: 95.25610564520399


##### Reducing the sparsity
To find the best model, we need to change and run the model multiple times with different hyperparameters.

Having 2 million reviews with 95 percent sparsity, the model will not have a clear result. Using random samples would create overfitting with smaller sample size and definitely would give very law accuracy on real data.

Training 2 million reviews with a matrix of 95 percent sparsity requires quite large gpus and big amount of time.

Because of that, we will reduce the sparsity by keeping the most popular 200 beers and their reviews.

When we find an optimum hyperparameters, we can add more beers.

In [25]:
#Get top 200 beers for model evaluation
top_beers = df.groupby('beer_id').size().reset_index(name='review_count')
top_beers = top_beers.sort_values(by='review_count', ascending=False).head(200)
#Update the existing dataframes
df = df.merge(top_beers, on='beer_id', how='inner')
df.reset_index(drop=True, inplace=True)
df_beers = df_beers.merge(top_beers, on='beer_id', how='inner')
df_beers.reset_index(drop=True, inplace=True)

#And print sparsity
print_sparsity_percentage(df, df_beers)

Number of Beers: 200
Number of Usernames: 4558
Number of Ratings: 308103
Sparsity Percentage: 66.20195261079421


#### Data types for tensorflow
Tensorflow does not accept int64 or float64 as data type so we need to do a little transformation here too.

In [26]:
# Convert int64 columns to int32
int_columns = df.select_dtypes(include=['int64']).columns
df[int_columns] = df[int_columns].astype('int32')

# Convert float64 columns to float32
float_columns = df.select_dtypes(include=['float64']).columns
df[float_columns] = df[float_columns].astype('float32')

#Do the same for df_beers
int_columns = df_beers.select_dtypes(include=['int64']).columns
df_beers[int_columns] = df_beers[int_columns].astype('int32')

# Convert float64 columns to float32
float_columns = df_beers.select_dtypes(include=['float64']).columns
df_beers[float_columns] = df_beers[float_columns].astype('float32')

print(df.info())
print(df_beers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308103 entries, 0 to 308102
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   beer_name          308103 non-null  object 
 1   beer_id            308103 non-null  int32  
 2   brewer_id          308103 non-null  int32  
 3   beer_abv           308103 non-null  float32
 4   beer_style         308103 non-null  object 
 5   review_appearance  308103 non-null  float32
 6   review_aroma       308103 non-null  float32
 7   review_palate      308103 non-null  float32
 8   review_taste       308103 non-null  float32
 9   review_overall     308103 non-null  float32
 10  review_time        308103 non-null  int32  
 11  review_username    308103 non-null  object 
 12  review_text        307302 non-null  object 
 13  review_count       308103 non-null  int32  
dtypes: float32(6), int32(4), object(4)
memory usage: 21.2+ MB
None
<class 'pandas.core.frame.DataFrame'>

#### End of Step 2
This is the end of step 2 where we made some cleaning and transformation. 

We will remove the csv file created in the previous step and create a new one with also csv file for beers only

In [27]:
import os
csv_file_path = os.path.join('dataset', 'ratebeer_dataset.csv')
df.to_csv(csv_file_path, index=False)
csv_beer_file_path = os.path.join('dataset', 'ratebeer_beers.csv')
df_beers.to_csv(csv_beer_file_path, index=False)